In [1]:
## Creating Project Folders

from pathlib import Path

project_root = Path(r"V:\Project\Project_Stick")

folders = [
    project_root / "data_source",
    project_root / "notebooks",
    project_root / "src",
    project_root / "outputs",
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folders created:")
for folder in folders:
    print(folder)

Project folders created:
V:\Project\Project_Stick\data_source
V:\Project\Project_Stick\notebooks
V:\Project\Project_Stick\src
V:\Project\Project_Stick\outputs


In [2]:
## Creating Project Folders

data_source = project_root / "data_source"

raw_folder = data_source / "raw"
processed_folder = data_source / "processed"

raw_folder.mkdir(parents=True, exist_ok=True)
processed_folder.mkdir(parents=True, exist_ok=True)

print(f"Created: {raw_folder}")
print(f"Created: {processed_folder}")

Created: V:\Project\Project_Stick\data_source\raw
Created: V:\Project\Project_Stick\data_source\processed


In [3]:
## Create a single DataFrame from multiple CSV files in the project
import pandas as pd
import re

frames = []
csv_files = sorted(raw_folder.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {raw_folder}")

for file in csv_files:
    file_df = pd.read_csv(file)
    file_df["Source_File"] = file.name

    year_match = re.search(r"20\d{2}", file.stem)
    file_df["Source_Year"] = (
        int(year_match.group()) if year_match else pd.NA
    )

    frames.append(file_df)

df = pd.concat(frames, ignore_index=True)

print(f"Combined {len(csv_files)} CSV files into one DataFrame.")
print(df.shape)
display(df.head())

Combined 3 CSV files into one DataFrame.
(1030, 11)


,S. No,Project Registration No.,Name and Address of the Promoter,Project Details and Address,Approval Details,Project Completion Date,Other Details,Form-c,Current Status of the Project,Source_File,Source_Year
0,1,TN/11/Building/0301/2024 ...,"Mayflower Enterprises Private Limited, Plot No...",Project Name: Mayflower East GateRegistration ...,"Member Secretary/ Joint Director (i/c), Coimba...",31.12.2026,Promoter Details ...,NaN,NaN,TNRERA_2024 .csv,2024
1,2,TN/29/Building/0302/2024 ...,"Ketan Mahendra Chandan, Flat No.Old No. 17, Ne...",Project Name: ACE STARLITStilt Floor + 4 Floor...,CMDA issued Planning Permission in Letter No. ...,Completed,Promoter Details ...,NaN,Completed,TNRERA_2024 .csv,2024
2,3,TN/16/Building/0303/2024 ...,"1. Thiru.V.Gouthaman and 2.M/s. Arjith Allure,...",Project Name: ARJITH ALLUREStilt Floor + 5 Fl...,The Joint Director / DTCP / Tiruchirappalli D...,13.11.2031,Promoter Details ...,NaN,NaN,TNRERA_2024 .csv,2024
3,4,TN/35/Building/0304/2024 ...,"M/s. Vinayaggas Infratech Private Limited, Plo...",Project Name: VINAYAGGAS - SAI NANDANAStilt Fl...,CMDA issued Planning Permission No. OL-PP/NHRB...,06.11.2031,Promoter Details ...,NaN,NaN,TNRERA_2024 .csv,2024
4,5,TN/29/Building/0305/2024 ...,"M/s.ETICADEVELOPERS PVT LTD, Door No.24, venka...",Project Name: ETICA MALAR MRC NAGARExtended Ba...,CMDA issued planning permission No: OL-PP/HRB/...,29.01.2032,Promoter Details ...,NaN,NaN,TNRERA_2024 .csv,2024


In [4]:
## Using Regex to extract specific information and creating new features from it. 

import re
import pandas as pd

## Step 1: Promoter Name ────────────────────────────────────────────────
df["Promoter Name"] = (
    df["Name and Address of the Promoter"].str.split(",").str[0].str.strip()
)

## Step 2: Project Name / Project Details / Project Address ────────────
def split_project(text):
    if pd.isna(text):
        return pd.Series(["", "", ""])
    if "Registration for" in text:
        name, rest = text.split("Registration for", 1)
        rest = "Registration for" + rest
    else:
        name, rest = "", text
    m = re.search(r'\s(at|comprised in)\s', rest)
    if m:
        details = rest[:m.start()].strip()
        address = rest[m.start():].strip()
    else:
        details, address = rest.strip(), ""
    return pd.Series([name.strip(), details, address])

df[["Project Name", "Project Details", "Project Address"]] = (
    df["Project Details and Address"].apply(split_project)
)

## Step 3: District / Lat / Long / coords_valid ─────────────────────────
district_source = (
    df["Project Details and Address"]
    .str.replace(r'[\u200b-\u200f\u202a-\u202e\ufeff]', '', regex=True)
)
district_pat = r'(?i)\b([A-Za-z]+)\s+(?:(?:Corporation|Municipality)\s*(?:&|/)?\s*)?District\b'
df["District"] = (
    district_source
    .str.extract(district_pat, expand=False)
    .str.strip()
    .str.title()
)

df["Latitude"] = df["Other Details"].str.extract(r'Latitude-([\d.]+)').astype(float)
df["Longitude"] = df["Other Details"].str.extract(r'Longitude-([\d.]+)').astype(float)

def normalize_coordinate(value):
    if pd.isna(value):
        return value
    if abs(value) >= 100:
        digits = str(int(abs(value)))
        value = int(digits[:2]) + int(digits[2:]) / 10 ** len(digits[2:])
    return value

df["Latitude"] = df["Latitude"].apply(normalize_coordinate)
df["Longitude"] = df["Longitude"].apply(normalize_coordinate)

reversed_coords = df["Latitude"].between(76, 81) & df["Longitude"].between(8, 14)
df.loc[reversed_coords, ["Latitude", "Longitude"]] = df.loc[
    reversed_coords, ["Longitude", "Latitude"]
].to_numpy()

df["coords_valid"] = df["Latitude"].between(8, 14) & df["Longitude"].between(76, 81)
pd.set_option("display.float_format", "{:.6f}".format)

## Step 4: Dwelling_Units / Heights_m / Floor_mentions ──────────────────
dwelling_count_patterns = [
    r'(?:total|totally)\s*(?:[-:]\s*)?(\d+)\s+(?:duplex\s+)?dwelling(?:\s+uni[s]?|s)?\b',
    r'(?:total|totally)\s*(?:[-:]\s*)?(\d+)\s+(?:EWS\s+)?tenements?\b',
    r'(?:total|totally)\s*(?:[-:]\s*)?(\d+)\s+dus\b',
    r'(?:total|totally)\s*(?:[-:]\s*)?(\d+)\s+(?:villa|flat|apartment|house)s?\b',
    r'(\d+)\s+(?:duplex\s+)?dwelling(?:\s+uni[s]?|s)?\b',
    r'(\d+)\s+(?:EWS\s+)?tenements?\b',
    r'(\d+)\s+dus\b',
    r'(\d+)\s+(?:villa|flat|apartment|house)s?\s+units?\b',
    r'(\d+)\s+(?:villa|flat|apartment|house)s?\b',
]

def extract_dwelling_units(text):
    text = str(text)
    if re.search(r'\bone\s+dwelling(?:\s+uni[s]?|s)?\b', text, flags=re.IGNORECASE):
        return 1
    for pattern in dwelling_count_patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            return int(match.group(1))
    return pd.NA

df["Dwelling_Units"] = (
    df["Project Details and Address"].apply(extract_dwelling_units).astype("Int64")
)

df["Heights_m"] = df["Project Details and Address"].apply(
    lambda t: [float(x) for x in re.findall(r'Height[^)]*?(\d+\.?\d*)\s*m', t, flags=re.I)]
)

floor_pat = r'Stilt\s+[Ff]loor[s]?(?:\s*\([^)]*\))?\s*\+\s*\d+\s*[Ff]loor[s]?'
df["Floor_mentions"] = df["Project Details and Address"].apply(
    lambda t: re.findall(floor_pat, str(t))
)

## Step 5: Building typology ────────────────────────────────────────────
residential_typology_pat = r'\b(residential|residencial|residence|residences|housing|dwelling|tenements?|villas?)\b'
commercial_typology_pat = (
    r'\bcommercial\b|\b(?:it\s*/\s*ites|ites)\b|'
    r'\bit\s+(?:building|park|office|complex)\b|'
    r'\b(?:office|offices|business|retail|shop|shops|shopping|showroom)s?\b'
)

def get_building_typology(text, dwelling_units):
    text = str(text)
    has_residential = bool(
        re.search(residential_typology_pat, text, flags=re.IGNORECASE)
        or pd.notna(dwelling_units)
    )
    has_commercial = bool(re.search(commercial_typology_pat, text, flags=re.IGNORECASE))
    has_industrial = bool(re.search(r'\bindustrial\b', text, flags=re.IGNORECASE))

    if has_residential and has_commercial:
        return "Residential, Commercial"
    if has_residential:
        return "Residential"
    if has_commercial:
        return "Commercial"
    if has_industrial:
        return "Industrial"
    return pd.NA

df["Building Typology"] = [
    get_building_typology(text, dwelling_units)
    for text, dwelling_units in zip(
        df["Project Details and Address"], df["Dwelling_Units"]
    )
]

## Step 6: Final column selection/order ─────────────────────────────────
final_cols = [
    "S. No", "Project Registration No.", "Project Completion Date",
    "Promoter Name", "Project Name", "Project Details", "Project Address",
    "District", "Latitude", "Longitude", "coords_valid",
    "Dwelling_Units", "Heights_m", "Floor_mentions", "Building Typology",
    "Source_File", "Source_Year",
]
df_clean = df[final_cols]

In [5]:
df_clean.head()

,S. No,Project Registration No.,Project Completion Date,Promoter Name,Project Name,Project Details,Project Address,District,Latitude,Longitude,coords_valid,Dwelling_Units,Heights_m,Floor_mentions,Building Typology,Source_File,Source_Year
0,1,TN/11/Building/0301/2024 ...,31.12.2026,Mayflower Enterprises Private Limited,,Project Name: Mayflower East GateRegistration ...,,Coimbatore,11.219200,77.113400,True,42,[],[Stilt floor +9 floors],Residential,TNRERA_2024 .csv,2024
1,2,TN/29/Building/0302/2024 ...,Completed,Ketan Mahendra Chandan,,Project Name: ACE STARLITStilt Floor + 4 Floor...,"at Old Door No. 15/1, New Door No. 31, Harring...",Chennai,13.040910,80.141200,True,<NA>,[],[Stilt Floor + 4 Floors],Commercial,TNRERA_2024 .csv,2024
2,3,TN/16/Building/0303/2024 ...,13.11.2031,1. Thiru.V.Gouthaman and 2.M/s. Arjith Allure,,Project Name: ARJITH ALLUREStilt Floor + 5 Fl...,"at Door No.13 , Arokiyasamy Pillai Street comp...",Tiruchirappalli,10.460000,78.400000,True,35,[],[Stilt Floor + 5 Floors],Residential,TNRERA_2024 .csv,2024
3,4,TN/35/Building/0304/2024 ...,06.11.2031,M/s. Vinayaggas Infratech Private Limited,,Project Name: VINAYAGGAS - SAI NANDANAStilt Fl...,"at Door No.23 & 26, Plot No.23 & 26, Thennanth...",Chengalpattu,12.574700,80.085900,True,20,[18.0],[Stilt Floor + 5 Floors],Residential,TNRERA_2024 .csv,2024
4,5,TN/29/Building/0305/2024 ...,29.01.2032,M/s.ETICADEVELOPERS PVT LTD,,Project Name: ETICA MALAR MRC NAGARExtended Ba...,"at Old Door No. 96, New Door No.140, Karpagam ...",Chennai,13.020911,80.269512,True,18,[],[],Residential,TNRERA_2024 .csv,2024


In [6]:
df.isnull().sum()

S. No                                  0
Project Registration No.               0
Name and Address of the Promoter       0
Project Details and Address            0
Approval Details                       1
Project Completion Date                0
Other Details                          0
Form-c                              1030
Current Status of the Project        812
Source_File                            0
Source_Year                            0
Promoter Name                          0
Project Name                           0
Project Details                        0
Project Address                        0
District                               0
Latitude                               0
Longitude                              0
coords_valid                           0
Dwelling_Units                       104
Heights_m                              0
Floor_mentions                         0
Building Typology                      8
dtype: int64

In [7]:
## Check Projects with missing Dwelling_Units for Residential Typology

residential_missing = df_clean[
    df_clean["Dwelling_Units"].isna()
    & df_clean["Building Typology"].eq("Residential")
]

display(residential_missing)

,S. No,Project Registration No.,Project Completion Date,Promoter Name,Project Name,Project Details,Project Address,District,Latitude,Longitude,coords_valid,Dwelling_Units,Heights_m,Floor_mentions,Building Typology,Source_File,Source_Year
34,35,TN/29/Building/0335/2024 ...,Completed,Meeradevi Chakravarthy,,"Stilt Floor + with 0 New Survey No. 32/1A2A2,O...","at comprised in Manapakkam Village, Alandur Ta...",Chennai,13.020682,18.183851,False,<NA>,[],[],Residential,TNRERA_2024 .csv,2024
48,49,TN/1/Building/0349/2024 ...,30.04.2029,M/s St Angelo's VNCT Ventures LLP,,Project Name: BROOKSIDE RESIDENCESRegistration...,"at Plot No. 8, 9 Southern Side Part, 9 Norther...",Kancheepuram,12.996730,80.113340,True,<NA>,[],[],Residential,TNRERA_2024 .csv,2024
54,55,TN/35/Building/0355/2024 ...,19.02.2027,M/s.Isha Homes (India) Pvt Ltd,,Project Name: “ISHA SYMPHONY VILLA”Registrati...,"comprised in S.Nos.157/13, 158/5 of Pudupakkam...",Chengalpattu,12.485800,80.123900,True,<NA>,[],[],Residential,TNRERA_2024 .csv,2024
153,154,TN/29/Building/0454/2024 ...,Completed,M/s.Tamil Nadu Urban Habitat Development Board,,Project Name: KARGIL NAGAR SCHEMEStilt Floor ...,"at comprised in Block No. 08,thiruvottiyur Vil...",Chennai,13.174678,80.292780,True,<NA>,[],[],Residential,TNRERA_2024 .csv,2024
187,188,TN/29/Building/0488/2024 ...,30.06.2029,R.Revathikumar,Project Name: SUPRABHATH,"Registration for Promoter's share of Flats , ...",at Stilt Floor + 4 Floors + 5 Floor (Part) re...,Chennai,12.985644,80.120000,True,<NA>,[],[Stilt Floor + 4 Floors],Residential,TNRERA_2024 .csv,2024
218,219,TN/10/Building/0519/2024 ...,18.09.2028,S.K.Senthil Kumar,Project Name: CRT Magilagam,Registration for Promoter's share of Flats 1st...,,Erode,11.335912,77.725072,True,<NA>,[],[Stilt Floor + 5 Floors],Residential,TNRERA_2024 .csv,2024
247,248,TN/29/Building/0548/2024 ...,30.06.2028,M/s.NUTECH REALTY PROJECT PVT LTD,,Project Name: Nutech Gardens Of GaiaRegistrati...,"comprised in S.No.180/2E2, 180/1C2A, 180/2F2, ...",Chennai,12.897635,80.244297,True,<NA>,[],[],Residential,TNRERA_2024 .csv,2024
271,272,TN/29/Building/0572/2024 ...,27.08.2032,M/s. Ramaniyam Realtors LLP,Project Name: RAMANIYAM PURNA KRISHNA,Registration for Promoter's Share of Flat Nos...,"at Door No.16/26 , New Bangaru Colony 1st Str...",Chennai,13.043566,80.191479,True,<NA>,[],[Stilt Floor + 5 Floors],Residential,TNRERA_2024 .csv,2024
354,42,TN/11/Building/0042/2025 ...,31.10.2029,RADIANCE REALTY DEVELOPERS INDIA LTD,Project Name: RADIANCE IMPERIA,Registration for Ground Floor + 2 Floors (Bloc...,,Coimbatore,11.006340,76.902100,True,<NA>,[],[],Residential,TNRERA_2025 .csv,2025
383,71,TN/29/Building/0071/2025 ...,25.11.2032,M/s. Ramaniyam Realtors LLP,Project Name: RAMANIYAM SRIDEVI,Registration for Promoter's share of Flat Nos....,"at New Door No.6 , Old Door No.5/2 , 4th Trus...",Chennai,13.026360,80.268750,True,<NA>,[],[Stilt Floor + 5 Floors],Residential,TNRERA_2025 .csv,2025


In [8]:
df_clean.shape

(1030, 17)

These rows can be dropped.

In [9]:
## Drop residential_missing from df_clean

df_clean = df_clean.drop(residential_missing.index)

df_clean.shape

(1011, 17)

In [10]:
df_clean.head(10)

,S. No,Project Registration No.,Project Completion Date,Promoter Name,Project Name,Project Details,Project Address,District,Latitude,Longitude,coords_valid,Dwelling_Units,Heights_m,Floor_mentions,Building Typology,Source_File,Source_Year
0,1,TN/11/Building/0301/2024 ...,31.12.2026,Mayflower Enterprises Private Limited,,Project Name: Mayflower East GateRegistration ...,,Coimbatore,11.219200,77.113400,True,42,[],[Stilt floor +9 floors],Residential,TNRERA_2024 .csv,2024
1,2,TN/29/Building/0302/2024 ...,Completed,Ketan Mahendra Chandan,,Project Name: ACE STARLITStilt Floor + 4 Floor...,"at Old Door No. 15/1, New Door No. 31, Harring...",Chennai,13.040910,80.141200,True,<NA>,[],[Stilt Floor + 4 Floors],Commercial,TNRERA_2024 .csv,2024
2,3,TN/16/Building/0303/2024 ...,13.11.2031,1. Thiru.V.Gouthaman and 2.M/s. Arjith Allure,,Project Name: ARJITH ALLUREStilt Floor + 5 Fl...,"at Door No.13 , Arokiyasamy Pillai Street comp...",Tiruchirappalli,10.460000,78.400000,True,35,[],[Stilt Floor + 5 Floors],Residential,TNRERA_2024 .csv,2024
3,4,TN/35/Building/0304/2024 ...,06.11.2031,M/s. Vinayaggas Infratech Private Limited,,Project Name: VINAYAGGAS - SAI NANDANAStilt Fl...,"at Door No.23 & 26, Plot No.23 & 26, Thennanth...",Chengalpattu,12.574700,80.085900,True,20,[18.0],[Stilt Floor + 5 Floors],Residential,TNRERA_2024 .csv,2024
4,5,TN/29/Building/0305/2024 ...,29.01.2032,M/s.ETICADEVELOPERS PVT LTD,,Project Name: ETICA MALAR MRC NAGARExtended Ba...,"at Old Door No. 96, New Door No.140, Karpagam ...",Chennai,13.020911,80.269512,True,18,[],[],Residential,TNRERA_2024 .csv,2024
5,6,TN/1/Building/0306/2024 ...,12.01.2032,Ms Isha Homes India Private Limited,,Project Name: ISHA POKKISHAMGroup Housing Deve...,"at 1st Floors) , Block B - Stilt Floor + 5 Fl...",Chengalpattu,12.540300,80.115200,True,33,[],"[Stilt Floor + 5 Floors, Stilt Floor + 5 Floors]",Residential,TNRERA_2024 .csv,2024
6,7,TN/29/Building/0307/2024 ...,21.09.2028,M/s.R.K.N Construction Proprietor Thiru.R.Radh...,,Project Name: NAVRANG FLATSRegistration of Sti...,"at Plot No.B, Gandhi street, Lakshmipuram, Thi...",Chennai,12.570000,80.070000,True,15,[],[Stilt Floor + 3 Floors],Residential,TNRERA_2024 .csv,2024
7,8,TN/29/Building/0308/2024 ...,Completed,T Kumar Mehra,,Project Name: INDUS ARBOURRegistration of Stil...,"at Old Door No.23, New Door No.9, Plot No.4363...",Chennai,13.050000,80.120000,True,1,[],[],"Residential, Commercial",TNRERA_2024 .csv,2024
8,9,TN/29/Building/0309/2024 ...,21.12.2031,Asset Tree Housing,,Project Name: ATH VALENCIARegistration of Stil...,"at Plot No.146, Defence Colony 12th Cross Stre...",Chennai,13.013000,80.120500,True,15,[18.3],[Stilt Floor + 5 Floors],Residential,TNRERA_2024 .csv,2024
9,10,TN/2/Building/0310/2024 ...,01.02.2026,Sri Hari Homes,,Project Name: MAGNUMRegistration of Stilt Floo...,"at Plot No. 11, Sri Balaji Avenue, Ayanambakka...",Thiruvallur,13.086698,80.146207,True,15,[12.0],[Stilt Floor + 3 Floors],Residential,TNRERA_2024 .csv,2024


In [11]:
# Validate district extraction and inspect the final unresolved rows
missing_district = df_clean[df_clean["District"].isna()]
print(f"Rows with missing districts: {len(missing_district)}")

display(
    df_clean.loc[
        df_clean["Project Registration No."].astype(str).str.contains(
            "0320/2024", regex=False
        ),
        ["Project Registration No.", "District"],
    ]
)

for index, row in df.loc[
    missing_district.index,
    ["Project Registration No.", "Project Details and Address"],
].iterrows():
    print(f"\n{index} | {row['Project Registration No.']}\n{row['Project Details and Address']}")

Rows with missing districts: 0


,Project Registration No.,District
19,TN/16/Building/0320/2024 ...,Trichy


In [12]:
missing_typology = df_clean[df_clean["Building Typology"].isna()]

dwelling_only_commercial = df_clean[
    df_clean["Dwelling_Units"].notna()
    & df_clean["Building Typology"].eq("Commercial")
]
mixed_projects = df_clean[df_clean["Building Typology"].eq("Residential, Commercial")]

print(f"Rows with missing building typology: {len(missing_typology)}")
print("Building typology counts:")
print(df_clean["Building Typology"].value_counts(dropna=False))
print(f"Mixed residential-commercial projects: {len(mixed_projects)}")
print(
    "Rows with dwelling units incorrectly labeled only Commercial: "
    f"{len(dwelling_only_commercial)}"
)

print("Example IT building:")
display(
    df_clean.loc[
        df_clean["Project Registration No."].astype(str).str.contains(
            "0317/2024", regex=False
        ),
        ["Project Registration No.", "Dwelling_Units", "Building Typology"],
    ]
)

Rows with missing building typology: 8
Building typology counts:
Building Typology
Residential                856
Residential, Commercial     74
Commercial                  73
<NA>                         8
Name: count, dtype: int64
Mixed residential-commercial projects: 74
Rows with dwelling units incorrectly labeled only Commercial: 0
Example IT building:


,Project Registration No.,Dwelling_Units,Building Typology
16,TN/35/Building/0317/2024 ...,<NA>,Commercial


In [13]:
 df_clean[df_clean["Building Typology"].isna()]


,S. No,Project Registration No.,Project Completion Date,Promoter Name,Project Name,Project Details,Project Address,District,Latitude,Longitude,coords_valid,Dwelling_Units,Heights_m,Floor_mentions,Building Typology,Source_File,Source_Year
337,25,TN/11/Building/0025/2025 ...,29.02.2028,M/S. RADIANT REAL PROPERTIES INDIA PRIVATE LIM...,,Registration for Ground Floor + 2 Floors Block...,,Coimbatore,11.047354,77.036961,True,<NA>,[],[],<NA>,TNRERA_2025 .csv,2025
370,58,TN/11/Building/0058/2025 ...,03.02.2030,CASAGRAND COVAAN PRIVATE LIMITED,,Project Name: CASAGRAND CELESTRegistration of ...,comprised in S.No. 524/2A & 524/2B of Sulur Vi...,Coimbatore,11.021755,78.129830,True,<NA>,[],[],<NA>,TNRERA_2025 .csv,2025
418,106,TN/35/Building/0106/2025 ...,13.02.2028,CASAGRAND PREMIER BUILDER LIMITED,,Project Name: CASAGRAND GOLDENGROVERegistrati...,at First Floor in the Owners use Plot in the a...,Chengalpattu,12.835604,80.179790,True,<NA>,[],[],<NA>,TNRERA_2025 .csv,2025
473,161,TN/35/Building/0161/2025 ...,31.12.2027,S&P FOUNDATION PVT LTD,Project Name: S&P NEW HAVEN RIBBON WALK,Registration for Wing - 2: Basement Floor + S...,"comprised in S.No. 76B/1, 77/1, 2A, 78/1A, 2, ...",Chengalpattu,12.836765,80.157297,True,<NA>,[],"[Stilt Floor + 16 Floors, Stilt Floor + 18 Flo...",<NA>,TNRERA_2025 .csv,2025
494,182,TN/11/Building/0182/2025 ...,31.12.2027,M/s Velmayil Developers,Project Name: SASHTI,Registration for Block Nos. 5 to 25 & 26 (club...,,Coimbatore,11.028304,77.013368,True,<NA>,[],[],<NA>,TNRERA_2025 .csv,2025
552,240,TN/1/Building/0240/2025 ...,20.08.2029,M/s St Angelo's VNCt Ventures LLP,Project Name: BEACH BOULEVARD PHASE 3,Registration for Group development of 7 blocks...,comprised in S.No.69/7 of Thiruvidanthai Villa...,Chengalpattu,12.725710,80.188320,True,<NA>,[],[],<NA>,TNRERA_2025 .csv,2025
672,360,TNRERA/29/BLG/0360/2025 ...,31.12.2026,M/s. THOLLOL ENTERPRISES AND A RAMASAMY,,Project Name: THOLLOL ENTERPRISESRegistration ...,"at Old Door No.45 & 45A, New Door No.165 & 166...",Chennai,13.047012,80.233032,True,<NA>,[],[Stilt Floor(part)+3 Floors],<NA>,TNRERA_2025 .csv,2025
976,236,TNRERA/35/BLG/0236/2026 ...,31.12.2031,ARUN EXCELLO HOMES PRIVATE LIMITED,Project Name: ZIVA MOHANAM,Registration for Ground floor-Block-1(1 to 16)...,,Chengalpattu,12.370840,80.093050,True,<NA>,[],[],<NA>,TNRERA_2026 .csv,2026


In [14]:
df_clean.shape

(1011, 17)

In [16]:
# Drop rows with missing building typology
df_clean = df_clean.dropna(subset=["Building Typology"])

print(f"Rows remaining: {len(df_clean)}")

df_clean.head()

Rows remaining: 1003


,S. No,Project Registration No.,Project Completion Date,Promoter Name,Project Name,Project Details,Project Address,District,Latitude,Longitude,coords_valid,Dwelling_Units,Heights_m,Floor_mentions,Building Typology,Source_File,Source_Year
0,1,TN/11/Building/0301/2024 ...,31.12.2026,Mayflower Enterprises Private Limited,,Project Name: Mayflower East GateRegistration ...,,Coimbatore,11.219200,77.113400,True,42,[],[Stilt floor +9 floors],Residential,TNRERA_2024 .csv,2024
1,2,TN/29/Building/0302/2024 ...,Completed,Ketan Mahendra Chandan,,Project Name: ACE STARLITStilt Floor + 4 Floor...,"at Old Door No. 15/1, New Door No. 31, Harring...",Chennai,13.040910,80.141200,True,<NA>,[],[Stilt Floor + 4 Floors],Commercial,TNRERA_2024 .csv,2024
2,3,TN/16/Building/0303/2024 ...,13.11.2031,1. Thiru.V.Gouthaman and 2.M/s. Arjith Allure,,Project Name: ARJITH ALLUREStilt Floor + 5 Fl...,"at Door No.13 , Arokiyasamy Pillai Street comp...",Tiruchirappalli,10.460000,78.400000,True,35,[],[Stilt Floor + 5 Floors],Residential,TNRERA_2024 .csv,2024
3,4,TN/35/Building/0304/2024 ...,06.11.2031,M/s. Vinayaggas Infratech Private Limited,,Project Name: VINAYAGGAS - SAI NANDANAStilt Fl...,"at Door No.23 & 26, Plot No.23 & 26, Thennanth...",Chengalpattu,12.574700,80.085900,True,20,[18.0],[Stilt Floor + 5 Floors],Residential,TNRERA_2024 .csv,2024
4,5,TN/29/Building/0305/2024 ...,29.01.2032,M/s.ETICADEVELOPERS PVT LTD,,Project Name: ETICA MALAR MRC NAGARExtended Ba...,"at Old Door No. 96, New Door No.140, Karpagam ...",Chennai,13.020911,80.269512,True,18,[],[],Residential,TNRERA_2024 .csv,2024


In [17]:
df_clean.isna().sum()

S. No                        0
Project Registration No.     0
Project Completion Date      0
Promoter Name                0
Project Name                 0
Project Details              0
Project Address              0
District                     0
Latitude                     0
Longitude                    0
coords_valid                 0
Dwelling_Units              77
Heights_m                    0
Floor_mentions               0
Building Typology            0
Source_File                  0
Source_Year                  0
dtype: int64

The 77 missing dwelling units account for 73 of commercial properties. + 3 which is unknown.

In [18]:
df_clean['Building Typology'].value_counts(dropna=False)

Building Typology
Residential                856
Residential, Commercial     74
Commercial                  73
Name: count, dtype: int64

From above analysis and observations we can see there are counts of
Residential                856
Residential, Commercial     74
Commercial                  73

This totals to 1003 properties that have been registered in Tamil Nadu at TNRERA in the years 2024,2025 and 2026.


